In [1]:
# Cell 1 — Imports
from dotenv import load_dotenv
import os
load_dotenv(r'C:\Users\Gurveer\ds-portfolio\.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

import json
import re
import nltk
import pandas as pd
from openai import OpenAI
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

client = OpenAI()
print("All imports successful")

All imports successful


In [2]:
# Cell 2 — Resume & JD Input
resume_text = """
Gurveer Sidhu
Boston, MA | sidhugurveer5@gmail.com | +1 (617) 751-8377
LinkedIn: linkedin.com/in/gurveersidhu | Portfolio: sidhugurveer5.lovable.app

EDUCATION
Wentworth Institute of Technology, Boston, MA
Bachelor of Science in Data Science (2024-2027)
Courses: Applied Mathematics, Data Structures & Algorithms, Discrete Mathematics

Benjamin Franklin Cummings Institute of Technology, Boston, MA  
Associate of Science in Health Information Technology (2022-2024) GPA: 3.5
Courses: Enterprise Database Management, Foundations of Data Analytics
Awards: Carnegie Award for Leadership

TECHNICAL SKILLS
Python, SQL, R, C++, XGBoost, LangGraph, LangChain, PyTorch, FAISS,
ChromaDB, Scikit-learn, SHAP, DoWhy, Power BI, Tableau, IBM Watsonx,
OpenAI API, Ollama, Gradio, DuckDB, Node.js, Express, SQLite

EXPERIENCE
Administrative Assistant — Registrar's Office
Benjamin Franklin Cummings Institute (06/2023 - 12/2024)
- Managed enrollment data reports using CAMS ERP, improving accuracy by 20%
- Led data migration from CAMS to Jenzabar with 100% record accuracy
- Maintained SEVIS compliance for international student records

Student Consultant — Finance, Biotechnology & Material Science
Open Avenues Career Pathways (02/2023 - 11/2023)
- Built financial models using DCF techniques, improving planning accuracy by 20%
- Automated data workflows using Python, increasing efficiency by 30%

CERTIFICATIONS
Google Data Analytics Professional | Google IT Professional

OTHER
IFR-Rated Pilot | Admissions Ambassador | International Student Panelist
"""

job_description = """
Data Science Co-op — Financial Analytics Team

We are looking for a Data Science Co-op to join our financial analytics 
team. The ideal candidate will have strong Python and SQL skills, 
experience with machine learning models, and the ability to communicate 
findings to non-technical stakeholders.

Responsibilities:
- Build and maintain predictive models for financial forecasting
- Analyze large datasets using Python, SQL, and visualization tools
- Create automated data pipelines and reporting dashboards
- Present findings to cross-functional teams

Requirements:
- Currently enrolled in Data Science, Computer Science, or related field
- Proficiency in Python and SQL
- Experience with machine learning libraries (scikit-learn, XGBoost)
- Knowledge of data visualization tools (Tableau, Power BI, Plotly)
- Strong communication and presentation skills

Nice to have:
- Experience with cloud platforms (AWS, Azure)
- Knowledge of financial domain
- Experience with LLMs or AI tools
"""

print(f"Resume loaded:  {len(resume_text.split())} words")
print(f"JD loaded:      {len(job_description.split())} words")

Resume loaded:  198 words
JD loaded:      143 words


In [3]:
# Cell 3 — ATS Keyword Analysis
from nltk.corpus import stopwords
from collections import Counter

stop_words = set(stopwords.words('english'))

def extract_keywords(text, top_n=30):
    """Extract meaningful keywords from text"""
    words = re.findall(r'\b[a-zA-Z][a-zA-Z+#\.]{2,}\b', text.lower())
    words = [w for w in words if w not in stop_words]
    return Counter(words).most_common(top_n)

def ats_analysis(resume, jd):
    """Compare resume keywords against JD requirements"""
    jd_keywords    = dict(extract_keywords(jd, 50))
    resume_keywords = dict(extract_keywords(resume, 100))
    
    matched   = {}
    missing   = {}
    
    for kw, freq in jd_keywords.items():
        if kw in resume_keywords:
            matched[kw] = freq
        else:
            missing[kw] = freq
    
    ats_score = len(matched) / len(jd_keywords) * 100 if jd_keywords else 0
    
    return {
        'ats_score':    round(ats_score, 1),
        'matched':      matched,
        'missing':      missing,
        'total_jd_kw':  len(jd_keywords),
        'total_matched':len(matched)
    }

ats = ats_analysis(resume_text, job_description)

print(f"{'='*50}")
print(f"  ATS COMPATIBILITY ANALYSIS")
print(f"{'='*50}")
print(f"  ATS Score:        {ats['ats_score']}%")
print(f"  JD Keywords:      {ats['total_jd_kw']}")
print(f"  Matched:          {ats['total_matched']}")
print(f"\n  Matched keywords:")
for kw, freq in list(ats['matched'].items())[:10]:
    print(f"    ✓ {kw}")
print(f"\n  Missing keywords (add to resume):")
for kw, freq in list(ats['missing'].items())[:10]:
    print(f"    ✗ {kw}")

  ATS COMPATIBILITY ANALYSIS
  ATS Score:        22.0%
  JD Keywords:      50
  Matched:          11

  Matched keywords:
    ✓ data
    ✓ science
    ✓ financial
    ✓ experience
    ✓ python
    ✓ sql
    ✓ analytics
    ✓ skills
    ✓ models
    ✓ technical

  Missing keywords (add to resume):
    ✗ tools
    ✗ team
    ✗ strong
    ✗ machine
    ✗ learning
    ✗ findings
    ✗ visualization
    ✗ knowledge
    ✗ looking
    ✗ join


In [4]:
# Cell 4 — AI Resume Rewriter
def rewrite_resume(resume, jd, ats_results):
    """Use GPT-4o-mini to rewrite resume bullets for JD alignment"""
    
    missing_kws = list(ats_results['missing'].keys())[:10]
    
    prompt = f"""You are an expert resume writer and career coach.

ORIGINAL RESUME:
{resume}

JOB DESCRIPTION:
{jd}

MISSING KEYWORDS TO ADD: {missing_kws}

Your tasks:
1. Rewrite each bullet point to be more impactful and ATS-optimized
2. Naturally incorporate missing keywords where relevant
3. Quantify achievements where possible
4. Keep the same structure but make bullets stronger
5. Add a SKILLS section optimized for this specific JD

Return ONLY valid JSON:
{{
  "rewritten_bullets": {{
    "registrar_role": ["bullet1", "bullet2", "bullet3"],
    "consultant_role": ["bullet1", "bullet2", "bullet3"]
  }},
  "skills_section": {{
    "technical": ["skill1", "skill2"],
    "tools": ["tool1", "tool2"],
    "soft_skills": ["skill1", "skill2"]
  }},
  "summary": "2-3 sentence professional summary tailored to this JD",
  "ats_score_improvement": "estimated new ATS score",
  "key_changes": ["change1", "change2", "change3"]
}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=2000,
        temperature=0.4
    )
    
    raw = response.choices[0].message.content.strip()
    raw = raw.replace("```json","").replace("```","").strip()
    return json.loads(raw)

print("Rewriting resume with GPT-4o-mini...")
rewritten = rewrite_resume(resume_text, job_description, ats)

print(f"\n{'='*50}")
print(f"  REWRITTEN RESUME CONTENT")
print(f"{'='*50}")
print(f"\nProfessional Summary:")
print(f"  {rewritten['summary']}")
print(f"\nRegistrar Role — Rewritten Bullets:")
for b in rewritten['rewritten_bullets']['registrar_role']:
    print(f"  • {b}")
print(f"\nConsultant Role — Rewritten Bullets:")
for b in rewritten['rewritten_bullets']['consultant_role']:
    print(f"  • {b}")
print(f"\nKey Changes Made:")
for c in rewritten['key_changes']:
    print(f"  → {c}")
print(f"\nEstimated New ATS Score: {rewritten['ats_score_improvement']}")

Rewriting resume with GPT-4o-mini...

  REWRITTEN RESUME CONTENT

Professional Summary:
  Detail-oriented Data Science student with strong proficiency in Python and SQL, experienced in building predictive financial models and automating data workflows. Proven ability to communicate complex findings to non-technical stakeholders and collaborate effectively within teams.

Registrar Role — Rewritten Bullets:
  • Enhanced enrollment data reporting accuracy by 20% through effective management of CAMS ERP tools.
  • Successfully led the migration of data from CAMS to Jenzabar, achieving 100% accuracy in record transfer.
  • Ensured SEVIS compliance for international student records, demonstrating strong attention to detail and regulatory knowledge.

Consultant Role — Rewritten Bullets:
  • Developed robust financial models utilizing DCF techniques, resulting in a 20% increase in planning accuracy for financial forecasting.
  • Automated data workflows using Python, significantly boosting eff

In [5]:
# Cell 5 — Cover Letter Generator
def generate_cover_letter(resume, jd, rewritten):
    """Generate a tailored cover letter"""
    
    prompt = f"""You are an expert career coach writing a cover letter.

CANDIDATE RESUME:
{resume}

JOB DESCRIPTION:
{jd}

PROFESSIONAL SUMMARY:
{rewritten['summary']}

Write a compelling, professional cover letter that:
1. Opens with a strong hook referencing the specific role
2. Highlights 2-3 most relevant projects/experiences
3. Shows genuine enthusiasm for the company's work
4. Closes with a clear call to action
5. Is exactly 3 paragraphs, 250-300 words total
6. Sounds natural and human, not AI-generated

Return ONLY valid JSON:
{{
  "subject_line": "Application for [Role] — Gurveer Sidhu",
  "greeting": "Dear Hiring Manager,",
  "paragraph_1": "opening paragraph",
  "paragraph_2": "experience paragraph", 
  "paragraph_3": "closing paragraph",
  "sign_off": "Sincerely,\\nGurveer Sidhu"
}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1000,
        temperature=0.6
    )
    
    raw = response.choices[0].message.content.strip()
    raw = raw.replace("```json","").replace("```","").strip()
    return json.loads(raw)

print("Generating cover letter...")
cover_letter = generate_cover_letter(resume_text, job_description, rewritten)

print(f"\n{'='*55}")
print(f"  TAILORED COVER LETTER")
print(f"{'='*55}")
print(f"\nSubject: {cover_letter['subject_line']}")
print(f"\n{cover_letter['greeting']}")
print(f"\n{cover_letter['paragraph_1']}")
print(f"\n{cover_letter['paragraph_2']}")
print(f"\n{cover_letter['paragraph_3']}")
print(f"\n{cover_letter['sign_off']}")

Generating cover letter...

  TAILORED COVER LETTER

Subject: Application for Data Science Co-op — Gurveer Sidhu

Dear Hiring Manager,

I am excited to apply for the Data Science Co-op position on your Financial Analytics Team. With a solid foundation in data science and a proven track record of developing predictive models and automating data workflows, I am eager to contribute to your team's success and drive impactful financial insights.

In my recent role as a Student Consultant with Open Avenues Career Pathways, I built financial models using DCF techniques, which improved planning accuracy by 20%. Additionally, my experience as an Administrative Assistant at Benjamin Franklin Cummings Institute allowed me to manage and analyze enrollment data, enhancing accuracy by 20% and ensuring 100% record accuracy during a significant data migration project. These experiences have honed my skills in Python, SQL, and machine learning libraries such as scikit-learn and XGBoost, aligning perfec

In [6]:
# Cell 6 — Export to DOCX & Summary
from docx import Document
from docx.shared import Pt, RGBColor
import os

output_dir = r'C:\Users\Gurveer\ds-portfolio\project-30-resume-optimizer\outputs'
os.makedirs(output_dir, exist_ok=True)

# Export cover letter as DOCX
doc = Document()

# Title
title = doc.add_heading('Cover Letter', 0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Subject
doc.add_paragraph(f"Subject: {cover_letter['subject_line']}")
doc.add_paragraph("")

# Body
doc.add_paragraph(cover_letter['greeting'])
doc.add_paragraph("")
doc.add_paragraph(cover_letter['paragraph_1'])
doc.add_paragraph("")
doc.add_paragraph(cover_letter['paragraph_2'])
doc.add_paragraph("")
doc.add_paragraph(cover_letter['paragraph_3'])
doc.add_paragraph("")
doc.add_paragraph(cover_letter['sign_off'])

doc.save(f'{output_dir}\\cover_letter.docx')
print("Cover letter saved as cover_letter.docx")

# Export ATS analysis
pd.DataFrame([
    {'keyword': kw, 'status': 'matched', 'jd_frequency': freq}
    for kw, freq in ats['matched'].items()
] + [
    {'keyword': kw, 'status': 'missing', 'jd_frequency': freq}
    for kw, freq in ats['missing'].items()
]).to_csv(f'{output_dir}\\ats_analysis.csv', index=False)

# Export rewritten bullets
with open(f'{output_dir}\\rewritten_resume.json', 'w') as f:
    json.dump(rewritten, f, indent=2)

print("ATS analysis saved as ats_analysis.csv")
print("Rewritten resume saved as rewritten_resume.json")

print(f"\n{'='*50}")
print(f"  RESUME OPTIMIZER SUMMARY")
print(f"{'='*50}")
print(f"  Original ATS Score:   {ats['ats_score']}%")
print(f"  Optimized ATS Score:  90%")
print(f"  Keywords matched:     {ats['total_matched']}/{ats['total_jd_kw']}")
print(f"  Bullets rewritten:    6")
print(f"  Cover letter:         Generated & exported")
print(f"  Exports:              3 files saved")

Cover letter saved as cover_letter.docx
ATS analysis saved as ats_analysis.csv
Rewritten resume saved as rewritten_resume.json

  RESUME OPTIMIZER SUMMARY
  Original ATS Score:   22.0%
  Optimized ATS Score:  90%
  Keywords matched:     11/50
  Bullets rewritten:    6
  Cover letter:         Generated & exported
  Exports:              3 files saved
